# Azure AD OAuth Setup for Snowflake
This notebook configures External OAuth with Azure Entra ID, creates the API role, and sets up a sample database.

In [ ]:
%%sql -r use_role
USE ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r drop_old_integrations
DROP SECURITY INTEGRATION IF EXISTS AZURE_OAUTH;
DROP SECURITY INTEGRATION IF EXISTS EXTERNAL_OAUTH_AZURE;

In [ ]:
%%sql -r create_integration
CREATE OR REPLACE SECURITY INTEGRATION AZURE_ENTRA_OAUTH
  TYPE = EXTERNAL_OAUTH
  ENABLED = TRUE
  EXTERNAL_OAUTH_TYPE = AZURE
  EXTERNAL_OAUTH_ISSUER = 'https://sts.windows.net/a66a44bf-bf04-4606-839d-3f956853233b/'
  EXTERNAL_OAUTH_JWS_KEYS_URL = 'https://login.microsoftonline.com/a66a44bf-bf04-4606-839d-3f956853233b/discovery/v2.0/keys'
  EXTERNAL_OAUTH_AUDIENCE_LIST = ('api://28c90a4e-4a96-4f78-ab0e-171bd1a984ba')
  EXTERNAL_OAUTH_TOKEN_USER_MAPPING_CLAIM = 'appid'
  EXTERNAL_OAUTH_SNOWFLAKE_USER_MAPPING_ATTRIBUTE = 'login_name'
  EXTERNAL_OAUTH_ANY_ROLE_MODE = 'ENABLE';

In [ ]:
%%sql -r create_role
CREATE ROLE IF NOT EXISTS SNOWFLAKE_API_ROLE;

In [ ]:
%%sql -r grant_role_to_user
GRANT ROLE SNOWFLAKE_API_ROLE TO USER API_SERVICE_USER;

In [ ]:
%%sql -r grant_usage
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE SNOWFLAKE_API_ROLE;
GRANT USAGE ON DATABASE "SNOWFLAKE_SAMPLE_Apps" TO ROLE SNOWFLAKE_API_ROLE;
GRANT USAGE ON SCHEMA "SNOWFLAKE_SAMPLE_Apps".PUBLIC TO ROLE SNOWFLAKE_API_ROLE;

In [ ]:
%%sql -r grant_select
GRANT SELECT ON ALL TABLES IN SCHEMA "SNOWFLAKE_SAMPLE_Apps".PUBLIC TO ROLE SNOWFLAKE_API_ROLE;
GRANT SELECT ON FUTURE TABLES IN SCHEMA "SNOWFLAKE_SAMPLE_Apps".PUBLIC TO ROLE SNOWFLAKE_API_ROLE;

In [ ]:
%%sql -r alter_user_defaults
ALTER USER API_SERVICE_USER SET DEFAULT_ROLE = SNOWFLAKE_API_ROLE;
ALTER USER API_SERVICE_USER SET DEFAULT_WAREHOUSE = COMPUTE_WH;

In [ ]:
%%sql -r set_login_name
-- Set login_name to the Azure AD CLIENT application ID
ALTER USER API_SERVICE_USER SET LOGIN_NAME = '89a80661-cf8b-4e10-b3f4-b2b06be53a81';

In [ ]:
%%sql -r create_db
CREATE DATABASE IF NOT EXISTS "SNOWFLAKE_SAMPLE_Apps";
USE DATABASE "SNOWFLAKE_SAMPLE_Apps";
USE SCHEMA PUBLIC;

In [ ]:
%%sql -r create_table
CREATE OR REPLACE TABLE EMPLOYEES (
    EMPLOYEE_ID INT AUTOINCREMENT PRIMARY KEY,
    FIRST_NAME VARCHAR(50),
    LAST_NAME VARCHAR(50),
    EMAIL VARCHAR(100),
    DEPARTMENT VARCHAR(50),
    JOB_TITLE VARCHAR(100),
    SALARY DECIMAL(10,2),
    HIRE_DATE DATE,
    IS_ACTIVE BOOLEAN DEFAULT TRUE
);

In [ ]:
%%sql -r insert_data
INSERT INTO EMPLOYEES (FIRST_NAME, LAST_NAME, EMAIL, DEPARTMENT, JOB_TITLE, SALARY, HIRE_DATE)
VALUES
    ('Rahul', 'Sharma', 'rahul.sharma@example.com', 'Engineering', 'Software Engineer', 95000.00, '2022-03-15'),
    ('Priya', 'Patel', 'priya.patel@example.com', 'Engineering', 'Senior Developer', 120000.00, '2021-01-10'),
    ('Amit', 'Kumar', 'amit.kumar@example.com', 'Sales', 'Sales Manager', 85000.00, '2020-07-22'),
    ('Sneha', 'Gupta', 'sneha.gupta@example.com', 'HR', 'HR Specialist', 72000.00, '2023-02-01'),
    ('Vikram', 'Singh', 'vikram.singh@example.com', 'Engineering', 'DevOps Engineer', 105000.00, '2021-11-08'),
    ('Ananya', 'Reddy', 'ananya.reddy@example.com', 'Marketing', 'Marketing Analyst', 78000.00, '2022-09-12'),
    ('Karthik', 'Nair', 'karthik.nair@example.com', 'Finance', 'Financial Analyst', 88000.00, '2020-04-30'),
    ('Deepika', 'Joshi', 'deepika.joshi@example.com', 'Engineering', 'QA Engineer', 82000.00, '2023-06-15'),
    ('Manish', 'Mishra', 'manish.mishra@example.com', 'Engineering', 'Tech Lead', 135000.00, '2019-08-20'),
    ('Kavita', 'Verma', 'kavita.verma@example.com', 'Sales', 'Account Executive', 76000.00, '2022-12-01');

In [ ]:
%%sql -r verify_data
SELECT * FROM EMPLOYEES;